In [1]:
# ==============================================================================
# CAPSTONE MASTER ORCHESTRATOR: FULL SENSITIVITY ANALYSIS
# UNCONSTRAINED VS ILLIQUIDITY GUARDED EXECUTIONS
# ==============================================================================
import pandas as pd
import numpy as np
import itertools
from pathlib import Path
from tqdm.auto import tqdm
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. PARAMETRIC PORTFOLIO CONFIGURATION
# ==============================================================================

# 1. Get the directory where the current Jupyter Notebook is located
# Path.cwd() gets the current working directory of the notebook
notebook_dir = Path.cwd() 

# 2. Navigate up to the parent directory (crypto-perpetual-arbitrage)
project_dir = notebook_dir.parent 

# 3. Navigate into the 'data' folder
DATA_DIR = project_dir / "data"


TOKENS_TO_EVALUATE = ['BTC', 'AVAX', 'BERA', 'KAITO']

EMP_START = pd.Timestamp('2025-03-01 00:00:00') 
EMP_END = pd.Timestamp('2025-12-31 23:59:00')
ALL_CALENDAR_DAYS = pd.date_range(start=EMP_START.date(), end=EMP_END.date(), freq='D').date
DAYS_TOTAL = (EMP_END - EMP_START).days

ROLLING_WINDOW_MINS = 1440  
TRADE_TIMEOUT_MINS = 300

FEES = {
    'binf': 1.70, 'okex': 1.75, 'byde': 3.20, 
    'kusw': 2.50, 'gate': 2.00, 'hype': 1.44
}

NAME_MAP = {
    'binance-futures': 'binf', 'bybit': 'byde', 'okex-swap': 'okex', 
    'gate-io-futures': 'gate', 'hyperliquid': 'hype', 'kucoin-futures': 'kusw'
}

# ------------------------------------------------------------------------------
# SENSITIVITY GRID
# ------------------------------------------------------------------------------
SIGMAS_TO_TEST = [2.0, 2.5, 3.0, 3.5, 4.0]
GUARDS_TO_TEST = [5.0, 10.0, 20.0]  # Representing 5%, 10%, 20% guard (in basis points)

# Build parameter combinations
CONFIG_RUNS = [{'sigma': s, 'guard': None} for s in SIGMAS_TO_TEST] + \
              [{'sigma': s, 'guard': g} for s in SIGMAS_TO_TEST for g in GUARDS_TO_TEST]

# ==============================================================================
# 2. NATIVE DATA UTILITIES 
# ==============================================================================
def robust_to_datetime(series):
    s_num = pd.to_numeric(series, errors='coerce')
    med_val = s_num.median()
    if pd.isna(med_val): return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_localize(None)
    unit = 'ns' if med_val > 1e16 else ('us' if med_val > 1e13 else ('ms' if med_val > 1e10 else 's'))
    return pd.to_datetime(s_num, unit=unit, errors='coerce', utc=True).dt.floor('1min').dt.tz_localize(None)

def load_aligned_bbo(df_master, exch_A, exch_B, bbo_paths):
    df_pair = pd.DataFrame(index=df_master.index)
    for exch in [exch_A, exch_B]:
        bbo = pd.read_csv(bbo_paths[exch], usecols=['minute', 'bid', 'ask'])
        bbo['timestamp'] = robust_to_datetime(bbo['minute'])
        bbo.set_index('timestamp', inplace=True)
        bbo = bbo[['bid', 'ask']].copy()
        
        bbo.loc[bbo['bid'] >= bbo['ask'], ['bid', 'ask']] = np.nan
        bbo = bbo.reindex(df_master.index).ffill(limit=3)
        df_pair[f'bid_{exch}'] = bbo['bid']
        df_pair[f'ask_{exch}'] = bbo['ask']
        df_pair[f'spread_{exch}'] = (np.log(bbo['ask']) - np.log(bbo['bid'])) * 10000

    df_pair['combined_spread'] = df_pair[f'spread_{exch_A}'] + df_pair[f'spread_{exch_B}']
    df_pair['liquidity_toll'] = df_pair['combined_spread'].rolling(1440).median().bfill()
    return df_pair

def calc_stochastic_ex_ante(df_master, adj_col, raw_col, w):
    df = pd.DataFrame(index=df_master.index)
    df[adj_col] = df_master[adj_col]
    df['raw_ma'] = df_master[raw_col].shift(1).rolling(w).mean()
    
    y_prev, x_prev = df[adj_col].shift(1), df[adj_col].shift(2)
    roll_cov = y_prev.rolling(w).cov(x_prev)
    roll_var_x = x_prev.rolling(w).var()
    roll_var_y = y_prev.rolling(w).var()
    mean_x, mean_y = x_prev.rolling(w).mean(), y_prev.rolling(w).mean()
    
    phi = (roll_cov / roll_var_x).clip(upper=0.999, lower=-0.999) 
    a = mean_y - phi * mean_x
    m = a / (1 - phi)
    var_e = (roll_var_y - (phi**2) * roll_var_x).clip(lower=1e-8)
    sigma_eq = np.sqrt(var_e / (1 - phi**2))
    
    df['s_score'] = (df[adj_col] - m) / sigma_eq
    df['kappa'] = np.where(phi > 0, -np.log(phi.clip(lower=1e-5)), 0)
    return df['s_score'].values, df['raw_ma'].values, df['kappa'].values

# ==============================================================================
# 3. METRIC CALCULATOR
# ==============================================================================
def compile_metrics(trade_log):
    tdf = pd.DataFrame(trade_log)
    if tdf.empty: 
        return 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
        
    tdf['close_date'] = pd.to_datetime(tdf['Exit_Time']).dt.date
    active_daily_bps = tdf.groupby('close_date')['Net_PnL'].sum()
    calendar_daily_bps = active_daily_bps.reindex(ALL_CALENDAR_DAYS, fill_value=0.0)
    
    cum_bps = calendar_daily_bps.sum()
    mdd_bps = (calendar_daily_bps.cumsum().cummax() - calendar_daily_bps.cumsum()).max()
    
    cal_sharpe = (calendar_daily_bps.mean() / max(calendar_daily_bps.std(), 1e-6)) * np.sqrt(365)
    act_sharpe = (active_daily_bps.mean() / max(active_daily_bps.std(), 1e-6)) * np.sqrt(365) if len(active_daily_bps) > 1 else 0.0
    trd_sharpe = tdf['Net_PnL'].mean() / max(tdf['Net_PnL'].std(), 1e-6)
    
    gross_gains = tdf.loc[tdf['Net_PnL'] > 0, 'Net_PnL'].sum()
    gross_losses = abs(tdf.loc[tdf['Net_PnL'] < 0, 'Net_PnL'].sum())
    g2p = (gross_gains / gross_losses) if gross_losses > 0 else 999.99

    return (
        len(tdf), 
        (tdf['Net_PnL'] > 0).mean() * 100, 
        cum_bps, 
        cal_sharpe, 
        act_sharpe, 
        trd_sharpe, 
        g2p, 
        tdf['Net_PnL'].min(), 
        mdd_bps, 
        tdf['Hold_Time'].mean()
    )

# ==============================================================================
# 4. ENGINE EXECUTION CORE
# ==============================================================================
global_results = []

for token in TOKENS_TO_EVALUATE:
    print(f"\n" + "="*120)
    print(f" ⚙️ PRE-COMPUTING ENGINE FOOTPRINTS FOR: {token}")
    print("="*120)
    
    master_parquet = DATA_DIR / f"{token}_TAGGED_MASTER.parquet"
    if not master_parquet.exists():
        print(f"[!] Warning: Master Parquet for {token} not found. Skipping.")
        continue
        
    df_master = pd.read_parquet(master_parquet).loc[EMP_START:EMP_END]
    
    bbo_paths = {
        'binf': DATA_DIR / rf"binf_{token}_bbo.csv",
        'byde': DATA_DIR / rf"byde_{token}_bbo.csv",
        'okex': DATA_DIR / rf"okex_{token}_bbo.csv",
        'gate': DATA_DIR / rf"gate_{token}_bbo.csv",
        'hype': DATA_DIR / rf"hype_{token}_bbo.csv",
        'kusw': DATA_DIR / rf"kusw_{token}_bbo.csv"
    }
    
    exchanges = list(bbo_paths.keys())
    pairs = list(itertools.combinations(exchanges, 2))
    
    signals, bbo_storage = {}, {}
    n_steps = len(df_master)
    timestamps = df_master.index
    
    def get_data_driven_epoch(exch_name, full_name):
        epoch_col = f'epoch_id_{full_name}'
        if epoch_col not in df_master.columns:
            epoch_col = f'epoch_id_{exch_name}'
        if epoch_col in df_master.columns:
            shift_mask = df_master[epoch_col] != df_master[epoch_col].shift(-1)
            shift_mask.iloc[-1] = False
            return shift_mask.values, shift_mask.sum()
        return np.zeros(n_steps, dtype=bool), 0

    # Cache calculations to ensure rapid iteration across different configs
    for exch_a, exch_b in tqdm(pairs, desc=f"Loading & Pre-computing Data", leave=False):
        pair_label = f"{exch_a.upper()}/{exch_b.upper()}"
        full_name_A = [k for k, v in NAME_MAP.items() if v == exch_a][0]
        full_name_B = [k for k, v in NAME_MAP.items() if v == exch_b][0]
        
        adj_col = f'spread_padj_idx_mid_avg_{full_name_A}_vs_{full_name_B}'
        raw_col = f'spread_raw_mid_{full_name_A}_vs_{full_name_B}'
        
        if adj_col not in df_master.columns or raw_col not in df_master.columns: continue
            
        bbo_df = load_aligned_bbo(df_master, exch_a, exch_b, bbo_paths)
        s_score, raw_ma, kappa = calc_stochastic_ex_ante(df_master, adj_col, raw_col, ROLLING_WINDOW_MINS)
        z_exec_raw = df_master[raw_col].values
        
        mask_a = (timestamps.minute == 59) if exch_a == 'hype' else get_data_driven_epoch(exch_a, full_name_A)[0]
        mask_b = (timestamps.minute == 59) if exch_b == 'hype' else get_data_driven_epoch(exch_b, full_name_B)[0]
        is_funding_epoch = mask_a | mask_b

        signals[pair_label] = {
            's_score': s_score, 'raw_ma': raw_ma, 'raw_mid': z_exec_raw,
            'is_funding': is_funding_epoch,
            'liquidity_toll': bbo_df['liquidity_toll'].values,
            'round_trip_fee_bps': (FEES[exch_a] + FEES[exch_b]) * 2
        }
        
        # Optimize BBO lookup via native vectors for gridsearch speed
        bbo_storage[pair_label] = {
            'bid_a': bbo_df[f'bid_{exch_a.lower()}'].values,
            'ask_a': bbo_df[f'ask_{exch_a.lower()}'].values,
            'bid_b': bbo_df[f'bid_{exch_b.lower()}'].values,
            'ask_b': bbo_df[f'ask_{exch_b.lower()}'].values,
            'df': bbo_df # Retain DataFrame explicitly for imputation logic
        }

    # ----------------------------------------------------------
    # SENSITIVITY GRID EXECUTION
    # ----------------------------------------------------------
    for config in tqdm(CONFIG_RUNS, desc=f"Evaluating Grid ({token})"):
        entry_threshold = config['sigma']
        max_entry_spread_bps = config['guard']
        
        active_positions = {}
        trade_log = []
        last_entry_t = {pair: -60 for pair in signals.keys()}

        for t in range(ROLLING_WINDOW_MINS, n_steps):
            current_time = timestamps[t]
            
            # --- 1. EVALUATE POSITION EXITS ---
            keys_to_remove = []
            for pair, pos in active_positions.items():
                sig_data = signals[pair]
                exch_a, exch_b = pair.split('/')
                
                hold_time = t - pos['entry_idx']
                timeout_hit = hold_time >= TRADE_TIMEOUT_MINS
                funding_hit = sig_data['is_funding'][t] 
                ma_reverted = (sig_data['raw_mid'][t] <= pos['target_ma']) if pos['direction'] == -1 else (sig_data['raw_mid'][t] >= pos['target_ma'])
                
                bid_a_val = bbo_storage[pair]['bid_a'][t]
                ask_a_val = bbo_storage[pair]['ask_a'][t]
                bid_b_val = bbo_storage[pair]['bid_b'][t]
                ask_b_val = bbo_storage[pair]['ask_b'][t]
                is_bbo_invalid = np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val)
                
                must_close = False
                exit_reason = None
                
                if funding_hit: must_close, exit_reason = True, "Funding_Epoch"
                elif timeout_hit: must_close, exit_reason = True, "Timeout"
                elif ma_reverted: must_close, exit_reason = True, "MA_Hit"
                elif pos.get('force_close_pending', False): must_close, exit_reason = True, pos.get('pending_reason')
                
                if must_close:
                    if is_bbo_invalid:
                        if hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                            valid_bbo = bbo_storage[pair]['df'].iloc[:t].dropna(subset=[
                                f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', 
                                f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'
                            ])
                            if not valid_bbo.empty:
                                last_valid = valid_bbo.iloc[-1]
                                bid_a_val, ask_a_val = last_valid[f'bid_{exch_a.lower()}'], last_valid[f'ask_{exch_a.lower()}']
                                bid_b_val, ask_b_val = last_valid[f'bid_{exch_b.lower()}'], last_valid[f'ask_{exch_b.lower()}']
                                
                                if exit_reason in ["Funding_Epoch", "Funding_Delayed"]:
                                    pos['pending_reason'] = "Funding_Imputed"
                                else:
                                    pos['pending_reason'] = "Timeout_Imputed"
                            else:
                                bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
                        else:
                            pos['force_close_pending'] = True
                            if 'pending_reason' not in pos:
                                pos['pending_reason'] = "MA_Hit_Delayed"
                            continue 
                    
                    if pos.get('force_close_pending', False) or (is_bbo_invalid and (hold_time >= TRADE_TIMEOUT_MINS or exit_reason in ["Funding_Epoch", "Funding_Delayed"])):
                        exit_reason = pos.get('pending_reason', "Timeout_Imputed")
                    elif funding_hit: exit_reason = "Funding_Epoch"
                    elif timeout_hit: exit_reason = "Timeout"
                    else: exit_reason = "MA_Hit"
                    
                    if pos['direction'] == -1:  
                        current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                        gross_pnl = pos['z_entry_exec'] - current_z_exec
                    else:                      
                        current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                        gross_pnl = current_z_exec - pos['z_entry_exec']
                    
                    net_pnl = gross_pnl - pos['round_trip_fee_bps']
                    
                    trade_log.append({
                        'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': current_time, 
                        'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': exit_reason
                    })
                    keys_to_remove.append(pair)
                    
            for k in keys_to_remove: del active_positions[k]

            # --- 2. EVALUATE POSITION ENTRIES ---
            for pair, sig_data in signals.items():
                if pair in active_positions or t - last_entry_t[pair] < 60 or sig_data['is_funding'][t]: continue  
                    
                obs_s = sig_data['s_score'][t]
                if np.isnan(obs_s) or np.abs(obs_s) < entry_threshold: continue
                    
                exch_a, exch_b = pair.split('/')
                bid_a_val = bbo_storage[pair]['bid_a'][t]
                ask_a_val = bbo_storage[pair]['ask_a'][t]
                bid_b_val = bbo_storage[pair]['bid_b'][t]
                ask_b_val = bbo_storage[pair]['ask_b'][t]
                
                if np.isnan(bid_a_val) or np.isnan(ask_a_val) or np.isnan(bid_b_val) or np.isnan(ask_b_val): continue
                    
                # ILLIQUIDITY ENTRY GUARD FILTER
                if max_entry_spread_bps is not None:
                    curr_spread_a = (np.log(ask_a_val) - np.log(bid_a_val)) * 10000
                    curr_spread_b = (np.log(ask_b_val) - np.log(bid_b_val)) * 10000
                    if (curr_spread_a + curr_spread_b) > max_entry_spread_bps:
                        continue

                direction = -1 if obs_s > 0 else 1
                target_ma = sig_data['raw_ma'][t]
                
                if direction == -1:
                    entry_phys = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                    gross_ev = entry_phys - target_ma
                else:
                    entry_phys = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                    gross_ev = target_ma - entry_phys
                    
                toll = sig_data['liquidity_toll'][t]
                net_ev = gross_ev - sig_data['round_trip_fee_bps'] - toll
                
                if net_ev <= 0: continue  
                    
                active_positions[pair] = {
                    'entry_idx': t, 'z_entry_exec': entry_phys, 
                    'target_ma': target_ma, 'direction': direction, 
                    'round_trip_fee_bps': sig_data['round_trip_fee_bps']
                }
                last_entry_t[pair] = t  

        # --- TERMINAL LIQUIDATION BLOCK ---
        terminal_idx = n_steps - 1
        terminal_time = timestamps[terminal_idx]
        
        for pair, pos in list(active_positions.items()):
            exch_a, exch_b = pair.split('/')
            hold_time = min(terminal_idx - pos['entry_idx'], TRADE_TIMEOUT_MINS)
            
            valid_bbo = bbo_storage[pair]['df'].dropna(subset=[f'bid_{exch_a.lower()}', f'ask_{exch_a.lower()}', f'bid_{exch_b.lower()}', f'ask_{exch_b.lower()}'])
            if not valid_bbo.empty:
                last_valid_row = valid_bbo.iloc[-1]
                bid_a_val, ask_a_val = last_valid_row[f'bid_{exch_a.lower()}'], last_valid_row[f'ask_{exch_a.lower()}']
                bid_b_val, ask_b_val = last_valid_row[f'bid_{exch_b.lower()}'], last_valid_row[f'ask_{exch_b.lower()}']
            else:
                bid_a_val, ask_a_val, bid_b_val, ask_b_val = 1.0, 1.0, 1.0, 1.0
                
            if pos['direction'] == -1:
                current_z_exec = (np.log(ask_a_val) - np.log(bid_b_val)) * 10000
                gross_pnl = pos['z_entry_exec'] - current_z_exec
            else:
                current_z_exec = (np.log(bid_a_val) - np.log(ask_b_val)) * 10000
                gross_pnl = current_z_exec - pos['z_entry_exec']
                
            net_pnl = gross_pnl - pos['round_trip_fee_bps']
            trade_log.append({
                'Pair': pair, 'Entry_Time': timestamps[pos['entry_idx']], 'Exit_Time': terminal_time,
                'Hold_Time': hold_time, 'Gross_PnL': gross_pnl, 'Net_PnL': net_pnl, 'Reason': "Terminal_Force_Clean"
            })

        # --- RECORD METRICS ---
        trades, win_pct, cum_pnl, c_sr, a_sr, t_sr, g2p, max_l, mdd, avg_h = compile_metrics(trade_log)
        
        guard_label = "Unconstrained" if max_entry_spread_bps is None else f"Guarded ({max_entry_spread_bps} bps)"
        
        global_results.append({
            'Asset': token,
            'Guard_Type': guard_label,
            'Sigma': entry_threshold,
            'Trades': trades,
            'Win_Rate_%': win_pct,
            'Cum_PnL_bps': cum_pnl,
            'Cal_Sharpe': c_sr,
            'Act_Sharpe': a_sr,
            'Trd_Sharpe': t_sr,
            'G2P': g2p,
            'Max_Loss_bps': max_l,
            'MDD_bps': mdd,
            'Avg_Hold_min': avg_h
        })

# ==============================================================================
# 5. FINAL SENSITIVITY OUTPUT 
# ==============================================================================
df_results = pd.DataFrame(global_results)

print("\n" + "="*150)
print(" 📊 FULL SENSITIVITY ANALYSIS RESULTS (DATAFRAME)")
print("="*150)
display(df_results)

print("\n\n" + "="*150)
print(" 📝 LATEX SENSITIVITY MATRIX (FOR ACADEMIC REPORTING)")
print("="*150)
latex_code = df_results.to_latex(
    index=False,
    float_format="%.2f",
    caption="Strategy Sensitivity Analysis across Thresholds and Illiquidity Guards",
    label="tab:full_sensitivity"
)
print(latex_code)
print("="*150)


 ⚙️ PRE-COMPUTING ENGINE FOOTPRINTS FOR: BTC


Loading & Pre-computing Data:   0%|          | 0/15 [00:00<?, ?it/s]

Evaluating Grid (BTC):   0%|          | 0/20 [00:00<?, ?it/s]


 ⚙️ PRE-COMPUTING ENGINE FOOTPRINTS FOR: AVAX


Loading & Pre-computing Data:   0%|          | 0/15 [00:00<?, ?it/s]

...

Loading & Pre-computing Data:   0%|          | 0/15 [00:00<?, ?it/s]

Evaluating Grid (KAITO):   0%|          | 0/20 [00:00<?, ?it/s]


 📊 FULL SENSITIVITY ANALYSIS RESULTS (DATAFRAME)


,Asset,Guard_Type,Sigma,Trades,Win_Rate_%,Cum_PnL_bps,Cal_Sharpe,Act_Sharpe,Trd_Sharpe,G2P,Max_Loss_bps,MDD_bps,Avg_Hold_min
0,BTC,Unconstrained,2.0,517,18.181818,-1413.618885,-4.583962,-7.726549,-0.342046,0.341496,-19.545404,1447.099262,27.750484
1,BTC,Unconstrained,2.5,357,20.728291,-625.573811,-2.037554,-3.558398,-0.134016,0.560810,-19.545404,878.711197,27.901961
2,BTC,Unconstrained,3.0,225,27.555556,-104.393206,-0.423778,-0.805847,-0.029462,0.872990,-19.545404,408.380370,28.822222
3,BTC,Unconstrained,3.5,162,36.419753,247.173940,1.149256,2.404302,0.084580,1.524845,-19.545404,133.085517,30.512346
4,BTC,Unconstrained,4.0,117,44.444444,432.657186,2.010618,5.078655,0.178022,2.534166,-19.545404,57.924464,31.521368
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,KAITO,Guarded (10.0 bps),3.5,244,58.606557,-433.363686,-0.447853,-0.952427,-0.042752,0.844399,-377.850218,1023.624049,50.745902
76,KAITO,Guarded (20.0 bps),3.5,333,52.552553,-1896.786507,-1.401847,-2.860294,-0.118877,0.614887,-444.607105,1896.786507,54.450450
77,KAITO,Guarded (5.0 bps),4.0,108,60.185185,166.550539,0.524939,1.253698,0.047548,1.168516,-166.353254,304.968947,50.287037
78,KAITO,Guarded (10.0 bps),4.0,201,58.706468,-120.481614,-0.139564,-0.305664,-0.015045,0.943879,-377.850218,951.711058,48.771144




 📝 LATEX SENSITIVITY MATRIX (FOR ACADEMIC REPORTING)
\begin{table}
\caption{Strategy Sensitivity Analysis across Thresholds and Illiquidity Guards}
\label{tab:full_sensitivity}
\begin{tabular}{llrrrrrrrrrrr}
\toprule
Asset & Guard_Type & Sigma & Trades & Win_Rate_% & Cum_PnL_bps & Cal_Sharpe & Act_Sharpe & Trd_Sharpe & G2P & Max_Loss_bps & MDD_bps & Avg_Hold_min \\
\midrule
BTC & Unconstrained & 2.00 & 517 & 18.18 & -1413.62 & -4.58 & -7.73 & -0.34 & 0.34 & -19.55 & 1447.10 & 27.75 \\
BTC & Unconstrained & 2.50 & 357 & 20.73 & -625.57 & -2.04 & -3.56 & -0.13 & 0.56 & -19.55 & 878.71 & 27.90 \\
BTC & Unconstrained & 3.00 & 225 & 27.56 & -104.39 & -0.42 & -0.81 & -0.03 & 0.87 & -19.55 & 408.38 & 28.82 \\
BTC & Unconstrained & 3.50 & 162 & 36.42 & 247.17 & 1.15 & 2.40 & 0.08 & 1.52 & -19.55 & 133.09 & 30.51 \\
BTC & Unconstrained & 4.00 & 117 & 44.44 & 432.66 & 2.01 & 5.08 & 0.18 & 2.53 & -19.55 & 57.92 & 31.52 \\
BTC & Guarded (5.0 bps) & 2.00 & 516 & 18.02 & -1369.66 & -4.21 & -7.03 &